In [1]:
import pandas as pd
from pymongo import MongoClient

# temizlenmis csv yi okuyorum
df = pd.read_csv('../data/processed/cleaned_tweets.csv')

# ONEMLI: Pandas NaN degerlerini Python None'a ceviriyorum
# boylece MongoDB'de duzgun "null" olarak saklanir
# bunu yapmazsam MongoDB'de "nan" string olarak kaydedilir ve sorgularda sorun cikarir
df = df.where(df.notna(), None)

# mongodb ye baglanma
client = MongoClient('mongodb://localhost:27017/')
print("MongoDB baglantisi basarili!")

# veritabani ve koleksiyon olustur
db = client['twitter_db_pipeline']
collection = db['tweetler']

# eger onceden veri varsa temizleyelim ki cift kayit olmasin
collection.delete_many({})

# pandas dataframe i dict listesine cevirip toplu yukleme (bulk insert)
# to_dict('records') her satiri bir sozluk (dict) yapar
# insert_many() hepsini tek seferde MongoDB'ye yazar — tek tek yazmaktan cok daha hizli
records = df.to_dict('records')
collection.insert_many(records)
print(f"{len(records)} dokuman MongoDB'ye yuklendi.")

MongoDB baglantisi basarili!
14601 dokuman MongoDB'ye yuklendi.


**MongoDB Sema Tasarimi:**
Verileri "Flat Document" (duz dokuman) seklinde tuttum. Her tweet bir JSON/BSON dokumani olarak kaydedildi. Tweet metni, havayolu, duygu durumu, tarih gibi hersey tek bir dokumanda. Tablolar arasi JOIN islemine gerek olmadigindan bu yapida okuma hizi cok iyi oluyor.

In [2]:
# $and: tum kosullarin AYNI ANDA saglanmasini ister
# projeksiyon (2. parametre): hangi alanlari getir hangileri getirme
# _id: 0 diyerek MongoDB'nin otomatik ekledigi _id alanini gizliyorum
# sort("retweet_count", -1): -1 azalan siralama demek
# limit(3): en fazla 3 sonuc getir
print("Sorgu 1: Virgin America + pozitif + guven=1.0:")
sonuc1 = collection.find(
    {"$and": [
        {"airline": "Virgin America"},
        {"airline_sentiment": "positive"},
        {"airline_sentiment_confidence": 1.0}
    ]},
    {"text": 1, "airline_sentiment_confidence": 1, "_id": 0}
).sort("retweet_count", -1).limit(3)
for doc in sonuc1:
    print(doc)

Sorgu 1: Virgin America + pozitif + guven=1.0:
{'airline_sentiment_confidence': 1.0, 'text': '@VirginAmerica Flying LAX to SFO and after looking at the awesome movie lineup I actually wish I was on a long haul.'}
{'airline_sentiment_confidence': 1.0, 'text': "Always have it together!!! You're welcome! RT @VirginAmerica: @jessicajaymes You're so welcome."}
{'airline_sentiment_confidence': 1.0, 'text': '@VirginAmerica thanks for gate checking my baggage on your full flight dfw-lax 883 and giving me early boarding too #sweet'}


In [3]:
# $or: kosullardan EN AZ BIRININ saglanmasi yeterli
# $gt: "greater than" yani "buyuktur" — 0.8'den buyuk olanlar gelir
print("Sorgu 2: Rotar VEYA iptal + guven > 0.8:")
sonuc2 = collection.find(
    {
        "$or": [
            {"negativereason": "Late Flight"},
            {"negativereason": "Cancelled Flight"}
        ],
        "airline_sentiment_confidence": {"$gt": 0.8}
    },
    {"airline": 1, "negativereason": 1, "airline_sentiment_confidence": 1, "_id": 0}
).limit(3)
for doc in sonuc2:
    print(doc)

Sorgu 2: Rotar VEYA iptal + guven > 0.8:
{'airline_sentiment_confidence': 1.0, 'negativereason': 'Late Flight', 'airline': 'Virgin America'}
{'airline_sentiment_confidence': 1.0, 'negativereason': 'Cancelled Flight', 'airline': 'Virgin America'}
{'airline_sentiment_confidence': 1.0, 'negativereason': 'Late Flight', 'airline': 'Virgin America'}


In [4]:
# $ne: "not equal" yani "esit degil" — "Bilinmiyor" olanlari disarida birak
# $exists: True — bu alan dokumanda var mi kontrol et
# skip(10): ilk 10 sonucu atla
# limit(3): sonraki 3 tanesini getir (2. sayfa gibi dusun)
print("Sorgu 3: Konum paylasan kullanicilar (sayfalama):")
sonuc3 = collection.find(
    {
        "tweet_location": {"$ne": "Bilinmiyor", "$exists": True},
        "airline_sentiment": "negative"
    },
    {"name": 1, "tweet_location": 1, "airline": 1, "_id": 0}
).skip(10).limit(3)
for doc in sonuc3:
    print(doc)

Sorgu 3: Konum paylasan kullanicilar (sayfalama):
{'airline': 'Virgin America', 'name': 'noelduan', 'tweet_location': 'SF â\x86\x94 NY'}
{'airline': 'Virgin America', 'name': 'gianagon', 'tweet_location': 'New York + Panama'}
{'airline': 'Virgin America', 'name': 'seimatrun', 'tweet_location': 'Los Angeles'}


In [5]:
# $group: belirtilen alana gore grupla (SQL'deki GROUP BY gibi)
# _id: "$airline" — airline alanina gore grupla
# $sum: 1 — her dokuman icin 1 ekle (yani say)
# $avg: ortalama hesapla
# $min: en kucuk degeri bul
# $max: en buyuk degeri bul
# $sort: toplam_tweet'e gore azalan sirala
print("Sorgu 4: Havayolu bazli istatistikler:")
pipeline1 = [
    {"$group": {
        "_id": "$airline",
        "toplam_tweet": {"$sum": 1},
        "ort_guven": {"$avg": "$airline_sentiment_confidence"},
        "min_guven": {"$min": "$airline_sentiment_confidence"},
        "max_rt": {"$max": "$retweet_count"}
    }},
    {"$sort": {"toplam_tweet": -1}}
]
for doc in collection.aggregate(pipeline1):
    print(doc)

Sorgu 4: Havayolu bazli istatistikler:
{'_id': 'United', 'toplam_tweet': 3822, 'ort_guven': 0.9008776818419676, 'min_guven': 0.335, 'max_rt': 7}
{'_id': 'US Airways', 'toplam_tweet': 2913, 'ort_guven': 0.9215784414692757, 'min_guven': 0.34, 'max_rt': 44}
{'_id': 'American', 'toplam_tweet': 2720, 'ort_guven': 0.916166911764706, 'min_guven': 0.3367, 'max_rt': 5}
{'_id': 'Southwest', 'toplam_tweet': 2420, 'ort_guven': 0.8865159504132231, 'min_guven': 0.3353, 'max_rt': 22}
{'_id': 'Delta', 'toplam_tweet': 2222, 'ort_guven': 0.8698782628262827, 'min_guven': 0.3363, 'max_rt': 31}
{'_id': 'Virgin America', 'toplam_tweet': 504, 'ort_guven': 0.8760861111111111, 'min_guven': 0.3482, 'max_rt': 4}


In [6]:
# $match: filtreleme (SQL'deki WHERE gibi) — sadece negatif tweetler
# $group: havayoluna gore grupla, sayi ve ortalama hesapla
# $project: cikti alanlarini belirle + yeni alan olustur
# $cond: if/then/else mantigi — ort_guven >= 0.9 ise "yuksek" degilse "dusuk"
# $round: ondalik basamak sayisini sinirla (3 basamak)
print("Sorgu 5: Negatif tweetlerde guven seviyesi etiketi:")
pipeline2 = [
    {"$match": {"airline_sentiment": "negative"}},
    {"$group": {
        "_id": "$airline",
        "negatif_sayisi": {"$sum": 1},
        "ort_guven": {"$avg": "$airline_sentiment_confidence"}
    }},
    {"$project": {
        "negatif_sayisi": 1,
        "ort_guven": {"$round": ["$ort_guven", 3]},
        "guven_seviyesi": {
            "$cond": {
                "if": {"$gte": ["$ort_guven", 0.9]},
                "then": "yuksek",
                "else": "dusuk"
            }
        }
    }},
    {"$sort": {"negatif_sayisi": -1}}
]
for doc in collection.aggregate(pipeline2):
    print(doc)

Sorgu 5: Negatif tweetlerde guven seviyesi etiketi:
{'_id': 'United', 'negatif_sayisi': 2633, 'ort_guven': 0.933, 'guven_seviyesi': 'yuksek'}
{'_id': 'US Airways', 'negatif_sayisi': 2263, 'ort_guven': 0.946, 'guven_seviyesi': 'yuksek'}
{'_id': 'American', 'negatif_sayisi': 1939, 'ort_guven': 0.944, 'guven_seviyesi': 'yuksek'}
{'_id': 'Southwest', 'negatif_sayisi': 1186, 'ort_guven': 0.921, 'guven_seviyesi': 'yuksek'}
{'_id': 'Delta', 'negatif_sayisi': 955, 'ort_guven': 0.902, 'guven_seviyesi': 'yuksek'}
{'_id': 'Virgin America', 'negatif_sayisi': 181, 'ort_guven': 0.902, 'guven_seviyesi': 'yuksek'}


In [7]:
# once toplam negatif tweet sayisini aliyorum — yuzde hesabi icin lazim
# $addFields: mevcut dokumana yeni alan ekler (mevcut alanlara dokunmaz)
# $divide: bolme islemi — adet / toplam
# $multiply: carpma — bolum sonucunu 100 ile carparak yuzdeye ceviriyorum
# $round: 1 ondalik basamaga yuvarla
toplam_negatif = collection.count_documents({"airline_sentiment": "negative"})
print("Sorgu 6: Sikayet nedenlerinin yuzdeleri:")
pipeline3 = [
    {"$match": {"airline_sentiment": "negative", "negativereason": {"$ne": "Belirtilmedi"}}},
    {"$group": {"_id": "$negativereason", "adet": {"$sum": 1}}},
    {"$addFields": {
        "yuzde": {
            "$round": [{"$multiply": [{"$divide": ["$adet", toplam_negatif]}, 100]}, 1]
        }
    }},
    {"$sort": {"adet": -1}},
    {"$limit": 5}
]
for doc in collection.aggregate(pipeline3):
    print(doc)

Sorgu 6: Sikayet nedenlerinin yuzdeleri:
{'_id': 'Customer Service Issue', 'adet': 2902, 'yuzde': 31.7}
{'_id': 'Late Flight', 'adet': 1660, 'yuzde': 18.1}
{'_id': "Can't Tell", 'adet': 1190, 'yuzde': 13.0}
{'_id': 'Cancelled Flight', 'adet': 843, 'yuzde': 9.2}
{'_id': 'Lost Luggage', 'adet': 721, 'yuzde': 7.9}


In [10]:
# createIndex(): belirtilen alan(lar) icin indeks olusturur
# [("airline", 1), ("airline_sentiment", 1)]: bilesik indeks
# 1 = artan siralama, -1 = azalan siralama
# explain(): sorgunun nasil calistigini gosterir — indeks kullandi mi kullanmadi mi
print("Sorgu 7: Bilesik indeks olusturuluyor...")
collection.create_index([("airline", 1), ("airline_sentiment", 1)])

# indeksin kullanildigini dogrula
aciklama = collection.find(
    {"airline": "Delta", "airline_sentiment": "negative"}
).explain()
kazanan_plan = aciklama['queryPlanner']['winningPlan']
if 'inputStage' in kazanan_plan:
    print("Kullanilan indeks:", kazanan_plan['inputStage'].get('indexName', 'bulunamadi'))
else:
    print("Plan detayi:", kazanan_plan)

Sorgu 7: Bilesik indeks olusturuluyor...
Kullanilan indeks: airline_1_airline_sentiment_1


In [11]:
# updateOne(): kosulu saglayan ILK dokumani gunceller
# $set: belirtilen alanlari ekler veya gunceller
# eger alan zaten varsa degerini degistirir, yoksa yeni alan olusturur
print("Sorgu 8: Viral tweet isaretleme:")
collection.update_one(
    {"retweet_count": {"$gt": 10}},
    {"$set": {
        "one_cikan": True,
        "kategori": "viral_tweet",
        "inceleme_notu": "yuksek etkilesim"
    }}
)
# guncellenen kaydi kontrol edelim
kontrol = collection.find_one(
    {"one_cikan": True},
    {"text": 1, "retweet_count": 1, "kategori": 1, "_id": 0}
)
print("Guncellenen kayit:", kontrol)

Sorgu 8: Viral tweet isaretleme:
Guncellenen kayit: {'retweet_count': 22, 'text': '@SouthwestAir beautiful day in Seattle! http://t.co/iqu0PPVq2S', 'kategori': 'viral_tweet'}


In [12]:
# updateMany(): kosulu saglayan TUM dokumanlari gunceller
# $set: yeni alan ekle
# $inc: mevcut sayisal alani belirtilen miktarda artir (increment)
# ikisini ayni anda yapabiliyorum — veritabanina 2 kere gitmeme gerek yok
print("Sorgu 9: Kesin sonuclari isaretle + RT artir:")
guncelleme = collection.update_many(
    {"airline_sentiment_confidence": 1.0},
    {
        "$set": {"kesin_sonuc": True},
        "$inc": {"retweet_count": 1}
    }
)
print(f"Guncellenen kayit sayisi: {guncelleme.modified_count}")

Sorgu 9: Kesin sonuclari isaretle + RT artir:
Guncellenen kayit sayisi: 10406


In [13]:
# deleteMany(): kosulu saglayan tum dokumanlari siler
# $lt: "less than" yani "kucuktur"
# $and ile iki kosulu birlikte kullaniyorum — ikisi de saglanmali
print("Sorgu 10: Dusuk guvenli + belirsiz kayitlari sil:")
silinen = collection.delete_many({
    "$and": [
        {"airline_sentiment_confidence": {"$lt": 0.5}},
        {"negativereason": "Can't Tell"}
    ]
})
print(f"Silinen kayit sayisi: {silinen.deleted_count}")

Sorgu 10: Dusuk guvenli + belirsiz kayitlari sil:
Silinen kayit sayisi: 25


In [17]:
# $regex: metin icinde desen arama (regular expression)
# "cancel|delay": cancel VEYA delay kelimesini ara
# $options: "i" = case-insensitive (buyuk/kucuk harf farketmez)
# $in: belirtilen degerlerden herhangi birine esit mi kontrol et
print("Sorgu 11: cancel/delay iceren United/Delta tweetleri:")
sonuc11 = collection.find(
    {
        "text": {"$regex": "cancel|delay", "$options": "i"},
        "airline": {"$in": ["United", "Delta"]}
    },
    {"airline": 1, "text": 1, "negativereason": 1, "_id": 0}
).sort("airline_sentiment_confidence", -1).limit(3)
for i, doc in enumerate(sonuc11, 1):
    print(f"\n--- Sonuc {i} ---")
    print(f"  Havayolu : {doc['airline']}")
    print(f"  Sebep    : {doc['negativereason']}")
    print(f"  Tweet    : {doc['text']}")

Sorgu 11: cancel/delay iceren United/Delta tweetleri:

--- Sonuc 1 ---
  Havayolu : Delta
  Sebep    : Cancelled Flight
  Tweet    : @JetBlue but by Cancelled Flighting my flight and pushing me to the next day I'd lose $150 hotel which was why I was trying to get a same-day flight.

--- Sonuc 2 ---
  Havayolu : Delta
  Sebep    : Cancelled Flight
  Tweet    : @JetBlue Cancelled Flighted my flight. Went with another airline 2 leave 2day. They Cancelled Flighted also. Called JetBlue &amp; got same flight but now $250 moreðº

--- Sonuc 3 ---
  Havayolu : Delta
  Sebep    : Late Flight
  Tweet    : @JetBlue why was Flight 1856 delayed to Buffalo ?  Itâs a direct flight and the plane is at the gate.


In [18]:
# bilesik _id: gruplama birden fazla alana gore yapilir
# {"havayolu": "$airline", "duygu": "$airline_sentiment"} seklinde
# her havayolu-duygu cifti icin ayri bir grup olusur
# ornegin: United-negative, United-positive, United-neutral
print("Sorgu 12: Buyuk 3 havayolunun duygu dagilimi:")
pipeline_ileri = [
    {"$match": {"airline": {"$in": ["United", "American", "US Airways"]}}},
    {"$group": {
        "_id": {"havayolu": "$airline", "duygu": "$airline_sentiment"},
        "adet": {"$sum": 1}
    }},
    {"$sort": {"_id.havayolu": 1, "adet": -1}}
]
for doc in collection.aggregate(pipeline_ileri):
    print(doc)

Sorgu 12: Buyuk 3 havayolunun duygu dagilimi:
{'_id': {'havayolu': 'American', 'duygu': 'negative'}, 'adet': 1939}
{'_id': {'havayolu': 'American', 'duygu': 'neutral'}, 'adet': 455}
{'_id': {'havayolu': 'American', 'duygu': 'positive'}, 'adet': 326}
{'_id': {'havayolu': 'US Airways', 'duygu': 'negative'}, 'adet': 2259}
{'_id': {'havayolu': 'US Airways', 'duygu': 'neutral'}, 'adet': 381}
{'_id': {'havayolu': 'US Airways', 'duygu': 'positive'}, 'adet': 269}
{'_id': {'havayolu': 'United', 'duygu': 'negative'}, 'adet': 2625}
{'_id': {'havayolu': 'United', 'duygu': 'neutral'}, 'adet': 697}
{'_id': {'havayolu': 'United', 'duygu': 'positive'}, 'adet': 492}


In [19]:
# $facet: tek sorguda birden fazla pipeline calistir
# her alt-pipeline bagimsiz calisir ve sonuclari ayri ayri doner
# bu sayede veritabanina 1 kere gidip 2 farkli analiz sonucu aliyorum
# performans acisindan cok verimli — 2 ayri sorgu gondermekten iyi
print("Sorgu 13: Tek sorguda cok boyutlu analiz ($facet):")
facet_pipeline = [
    {"$facet": {
        "duygu_dagilimi": [
            {"$group": {"_id": "$airline_sentiment", "sayi": {"$sum": 1}}},
            {"$sort": {"sayi": -1}}
        ],
        "en_yaygin_sikayet": [
            {"$match": {"negativereason": {"$ne": "Belirtilmedi"}}},
            {"$group": {"_id": "$negativereason", "sayi": {"$sum": 1}}},
            {"$sort": {"sayi": -1}},
            {"$limit": 3}
        ]
    }}
]
sonuc = list(collection.aggregate(facet_pipeline))
print("Duygu dagilimi:", sonuc[0]['duygu_dagilimi'])
print("En yaygin 3 sikayet:", sonuc[0]['en_yaygin_sikayet'])

Sorgu 13: Tek sorguda cok boyutlu analiz ($facet):
Duygu dagilimi: [{'_id': 'negative', 'sayi': 9132}, {'_id': 'neutral', 'sayi': 3091}, {'_id': 'positive', 'sayi': 2353}]
En yaygin 3 sikayet: [{'_id': 'Customer Service Issue', 'sayi': 2902}, {'_id': 'Late Flight', 'sayi': 1660}, {'_id': "Can't Tell", 'sayi': 1165}]


In [20]:
# $bucket: sayisal bir alani belirtilen araliklara boler
# groupBy: hangi alana gore bolunecek
# boundaries: aralik sinirlari — [0, 0.4, 0.7, 0.9, 1.01] demek:
#   0-0.4 arasi, 0.4-0.7 arasi, 0.7-0.9 arasi, 0.9-1.01 arasi
# default: sinirlarin disinda kalan degerler icin
# output: her aralik icin ne hesaplanacak
print("Sorgu 14: Guven skoru aralik analizi ($bucket):")
bucket_pipeline = [
    {"$bucket": {
        "groupBy": "$airline_sentiment_confidence",
        "boundaries": [0, 0.4, 0.7, 0.9, 1.01],
        "default": "Diger",
        "output": {
            "tweet_sayisi": {"$sum": 1},
            "ornek_havayolu": {"$first": "$airline"}
        }
    }}
]
for doc in collection.aggregate(bucket_pipeline):
    print(doc)

Sorgu 14: Guven skoru aralik analizi ($bucket):
{'_id': 0, 'tweet_sayisi': 211, 'ornek_havayolu': 'Virgin America'}
{'_id': 0.4, 'tweet_sayisi': 3636, 'ornek_havayolu': 'Virgin America'}
{'_id': 0.7, 'tweet_sayisi': 310, 'ornek_havayolu': 'Virgin America'}
{'_id': 0.9, 'tweet_sayisi': 10419, 'ornek_havayolu': 'Virgin America'}


In [22]:
# distinct(): belirtilen alandaki benzersiz (tekil) degerleri dondurur
# SQL'deki SELECT DISTINCT karsiligi
# ornegin 14.000 tweette kac farkli sehir var?
print("Sorgu 15: Benzersiz deger analizi (distinct):")
benzersiz_konumlar = collection.distinct("tweet_location")
# "Bilinmiyor" olanlari cikariyorum
gercek_konumlar = [k for k in benzersiz_konumlar if k != "Bilinmiyor" and k is not None]
print(f"Toplam benzersiz konum sayisi: {len(gercek_konumlar)}")
print(f"Ilk 5 konum: {gercek_konumlar[:5]}")

# her havayoluna ait benzersiz sikayet sebebi sayisi
for havayolu in ["United", "Delta", "Southwest"]:
    sebepler = collection.distinct("negativereason", {"airline": havayolu})
    print(f"{havayolu}: {len(sebepler)} farkli sikayet sebebi")

Sorgu 15: Benzersiz deger analizi (distinct):
Toplam benzersiz konum sayisi: 3074
Ilk 5 konum: ['  || san antonio, texas||', ' Bronx, NY / Destin, Fl', ' California 92705', ' D(MD)V â\x9c\x88ï¸\x8f NYC â\x9c\x88ï¸\x8f Germany', ' DC | Jersey City']
United: 11 farkli sikayet sebebi
Delta: 11 farkli sikayet sebebi
Southwest: 11 farkli sikayet sebebi
